# Búsqueda nocturna de columnas (subconjuntos × 5 redes)

Busca las **mejores columnas** (no la mejor red) para predecir el retorno de ETH.

**Las dos cajas:** se saca un `k` uniforme en `[K_MIN, K_MAX]` (cuántas columnas) y luego `k` columnas al azar.
El mismo *conjunto* exacto de columnas nunca se repite; el mismo `k` con columnas distintas sí.

Cada subconjunto se entrena en **5 redes** de complejidad creciente (de *underfit* a *overfit*).
El ranking es por **val_loss medio entre las 5 redes**: media baja + `std` bajo = subconjunto **robusto** (señal).
Si solo brilla en una red, el `std` lo delata (banda de ruido del DirAcc ~47-53%). El **test (15% final) NO se toca aquí**.

**Cómo usarlo:**
1. Ejecuta las celdas de configuración, datos y funciones.
2. Ejecuta la celda *Inicializar estado* **una vez**.
3. Ejecuta la celda *Bucle de búsqueda*. Para **parar**: botón **Interrupt (■)** del kernel.
   Para **reanudar** donde lo dejaste: vuelve a ejecutar esa misma celda (no reinicialices el estado).
4. Celda de *análisis* al final.

Cada resultado se vuelca al CSV al instante, así que un corte no pierde lo hecho.

In [1]:
# === Configuración =========================================================
import os, csv, time, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import RobustScaler, StandardScaler

CSV_PATH    = "../data/csv/df_model.csv"   # df_model COMPLETO (~2800 x ~84)
OUT_DETALLE = "resultados_busqueda.csv"             # una fila por (subconjunto x red)
TARGET_COL  = "eth_close_ret"

SEQ_LEN, HORIZON = 30, 3
K_MIN, K_MAX     = 10, 55

# --- Definición del TARGET --------------------------------------------------
# ¡VERIFICA que coincide con tu notebook! Es lo único que no puedo ver.
#   "h_ahead" -> retorno DIARIO de dentro de HORIZON días = ret.shift(-HORIZON)
#   "cum_h"   -> retorno ACUMULADO de los próximos HORIZON días
TARGET_MODE = "h_ahead"

# --- Entrenamiento (receta para series financieras ruidosas) ----------------
MAX_EPOCHS, ES_PATIENCE = 60, 8
BATCH_SIZE, LR, WEIGHT_DECAY = 64, 1e-3, 1e-4
HUBER_DELTA, GRAD_CLIP = 1.0, 1.0
SCHED_FACTOR, SCHED_PATIENCE = 0.5, 3
SEED = 42                 # semilla FIJA: lo único que varía es columnas y arquitectura

# --- Búsqueda ---------------------------------------------------------------
MAX_TRIALS     = None     # None = infinito (paras con Interrupt)
TOP_K          = 25
PROGRESS_EVERY = 10
MAX_RESAMPLE   = 200

EXCLUIR = {
    "btc_open","btc_high","btc_low","btc_close","btc_volume",
    "eth_open","eth_high","eth_low","eth_close","eth_volume",
    "btc_mcap","eth_mcap","fear_greed","inflation","fed_rate",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cuda


In [2]:
# === Las 5 redes (escalera de complejidad: underfit -> overfit) ============
# hidden = tamaños de capas LSTM apiladas (p.ej. [32,16] = 32->16). Dropout entre capas y antes del FC.
REDES = [
    {"id": 1, "name": "minima",  "hidden": [16],     "dropout": 0.20},  # suelo / underfit
    {"id": 2, "name": "pequena", "hidden": [32],     "dropout": 0.30},  # zona sana baja
    {"id": 3, "name": "media",   "hidden": [32, 16], "dropout": 0.35},  # candidata (Occam)
    {"id": 4, "name": "grande",  "hidden": [48, 24], "dropout": 0.35},  # la actual
    {"id": 5, "name": "maxima",  "hidden": [64, 32], "dropout": 0.40},  # techo / overfit
]

In [3]:
# === Datos: carga, pool de features, escalado (fit solo con train) =========
def construir_target(df):
    r = df[TARGET_COL]
    if TARGET_MODE == "h_ahead":
        return r.shift(-HORIZON)                      # retorno diario de dentro de HORIZON días
    elif TARGET_MODE == "cum_h":
        return sum(r.shift(-(i + 1)) for i in range(HORIZON))   # acumulado próximos HORIZON días
    raise ValueError(TARGET_MODE)

def cargar_datos():
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"No encuentro {CSV_PATH}. Genera df_model_completo.csv (df_model COMPLETO).")
    df = pd.read_csv(CSV_PATH)
    if "date" in df.columns:
        df = df.set_index("date")
    y_full = construir_target(df).astype("float32")
    pool = [c for c in df.columns if c not in EXCLUIR and c != TARGET_COL]
    X_full = df[pool].astype("float32")
    # quitar NaN UNA vez sobre todo el pool -> todos los subconjuntos comparten filas (comparación justa)
    mask = X_full.notna().all(axis=1) & y_full.notna()
    X_full, y_full = X_full[mask], y_full[mask]
    n = len(X_full)
    print(f"[datos] {n} filas usables x {len(pool)} columnas candidatas "
          f"(de {df.shape[1]} totales; {len(EXCLUIR)} excluidas + target).")
    # binarias 0/1 (one-hot regimen y Fear&Greed): NO se escalan (IQR=0 -> inf/nan)
    es_bin = np.array([set(np.unique(X_full[c].values)).issubset({0.0, 1.0}) for c in pool])
    cols_escalar = [c for c, b in zip(pool, es_bin) if not b]
    print(f"[datos] {len(cols_escalar)} continuas a escalar, {int(es_bin.sum())} binarias (sin escalar).")
    # split temporal 70/15/15 a nivel de SECUENCIAS
    N = n - SEQ_LEN + 1
    n_train, n_val = int(N * 0.70), int(N * 0.15)
    fila_fin_train = (n_train - 1) + (SEQ_LEN - 1)
    Xtr_fit = X_full.iloc[:fila_fin_train + 1]
    ytr_fit = y_full.iloc[:fila_fin_train + 1]
    rscaler = RobustScaler().fit(Xtr_fit[cols_escalar].values)
    X_scaled = X_full.copy()
    X_scaled[cols_escalar] = rscaler.transform(X_full[cols_escalar].values).astype("float32")
    tscaler = StandardScaler().fit(ytr_fit.values.reshape(-1, 1))
    y_scaled = tscaler.transform(y_full.values.reshape(-1, 1)).astype("float32").ravel()
    print(f"[split] {N} secuencias -> train {n_train} | val {n_val} | "
          f"test {N - n_train - n_val} (test reservado, no se toca).")
    return {"pool": pool, "X": X_scaled.values.astype("float32"), "y": y_scaled,
            "tscaler": tscaler, "N": N, "n_train": n_train, "n_val": n_val,
            "col_idx": {c: i for i, c in enumerate(pool)}}

def construir_secuencias(X_sub, y, seq_len):
    sw = np.lib.stride_tricks.sliding_window_view(X_sub, seq_len, axis=0)  # (N, F, seq_len)
    return np.transpose(sw, (0, 2, 1)).copy(), y[seq_len - 1:]

info = cargar_datos()
pool, X, y = info["pool"], info["X"], info["y"]
N, n_train, n_val = info["N"], info["n_train"], info["n_val"]
tscaler, col_idx = info["tscaler"], info["col_idx"]

[datos] 2836 filas usables x 83 columnas candidatas (de 99 totales; 15 excluidas + target).
[datos] 75 continuas a escalar, 8 binarias (sin escalar).
[split] 2807 secuencias -> train 1964 | val 421 | test 422 (test reservado, no se toca).


In [4]:
# === Modelo: LSTM apilada con tamaños arbitrarios ==========================
class LSTMStack(nn.Module):
    def __init__(self, n_features, hidden_sizes, dropout):
        super().__init__()
        self.lstms = nn.ModuleList()
        in_size = n_features
        for h in hidden_sizes:
            self.lstms.append(nn.LSTM(in_size, h, batch_first=True))
            in_size = h
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_sizes[-1], 1)
    def forward(self, x):
        for i, lstm in enumerate(self.lstms):
            x, _ = lstm(x)
            if i < len(self.lstms) - 1:
                x = self.drop(x)                 # dropout entre capas
        return self.head(self.drop(x[:, -1, :])).squeeze(-1)

def contar_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [5]:
# === Entrenamiento de una red ==============================================
def entrenar_red(red_cfg, Xtr, ytr, Xva, yva, tscaler):
    torch.manual_seed(SEED); np.random.seed(SEED)   # init reproducible: solo varían columnas y red
    model = LSTMStack(Xtr.shape[2], red_cfg["hidden"], red_cfg["dropout"]).to(DEVICE)
    n_params = contar_params(model)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min",
                                                       factor=SCHED_FACTOR, patience=SCHED_PATIENCE)
    loss_fn = nn.HuberLoss(delta=HUBER_DELTA)
    Xtr_t = torch.from_numpy(Xtr).to(DEVICE); ytr_t = torch.from_numpy(ytr).to(DEVICE)
    Xva_t = torch.from_numpy(Xva).to(DEVICE); yva_t = torch.from_numpy(yva).to(DEVICE)
    n = Xtr_t.shape[0]
    best_val, best_epoch, best_diracc, sin_mejora = float("inf"), -1, float("nan"), 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            opt.zero_grad()
            loss = loss_fn(model(Xtr_t[idx]), ytr_t[idx])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        model.eval()
        with torch.no_grad():
            pv = model(Xva_t)
            vloss = loss_fn(pv, yva_t).item()
        sched.step(vloss)
        if vloss < best_val - 1e-6:
            best_val, best_epoch, sin_mejora = vloss, epoch, 0
            # DirAcc en espacio de retorno ORIGINAL (StandardScaler mueve el 0)
            pv_ret = tscaler.inverse_transform(pv.cpu().numpy().reshape(-1, 1)).ravel()
            yv_ret = tscaler.inverse_transform(yva_t.cpu().numpy().reshape(-1, 1)).ravel()
            best_diracc = float(np.mean((pv_ret > 0) == (yv_ret > 0)))
        else:
            sin_mejora += 1
            if sin_mejora >= ES_PATIENCE:
                break
    del Xtr_t, ytr_t, Xva_t, yva_t, model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return best_val, best_diracc, n_params, best_epoch

In [6]:
# === Utilidades: hash, guardado, leaderboard ===============================
def hash_conjunto(cols):
    return hashlib.md5("|".join(sorted(cols)).encode()).hexdigest()

def guardar_fila(row):
    nuevo = not os.path.exists(OUT_DETALLE)
    with open(OUT_DETALLE, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["timestamp","trial","set_hash","k","red_id","red_name",
                        "n_params","val_loss","val_diracc","best_epoch","columnas"])
        w.writerow(row)

def actualizar_leaderboard(lb, reg):
    lb.append(reg); lb.sort(key=lambda r: r["mean_val_loss"]); del lb[TOP_K:]

def imprimir_leaderboard(lb, n=None):
    n = n or len(lb)
    print("\n" + "=" * 100)
    print(f"  LEADERBOARD — top {min(n, len(lb))} por VAL_LOSS MEDIO entre las 5 redes")
    print("  (std bajo = rinde parecido en todas las redes = robusto)")
    print("=" * 100)
    print(f"  {'#':>2}  {'k':>3}  {'val_loss medio':>14}  {'std':>7}  {'mejor red':>9}  "
          f"{'val_loss min':>12}  {'DirAcc med':>10}")
    print("  " + "-" * 96)
    for i, r in enumerate(lb[:n], 1):
        print(f"  {i:>2}  {r['k']:>3}  {r['mean_val_loss']:>14.6f}  {r['std_val_loss']:>7.4f}  "
              f"{r['best_net']:>9}  {r['min_val_loss']:>12.6f}  {r['mean_diracc']:>10.4f}")
    print("=" * 100)
    if lb:
        print(f"\n  Columnas del nº1 (k={lb[0]['k']}):")
        for c in sorted(lb[0]["columnas"]):
            print("     -", c)
    print()

### Inicializar estado — ejecuta **una sola vez**
Re-ejecútala solo si quieres empezar la búsqueda **de cero** (resetea contador y conjuntos vistos).

In [7]:
leaderboard = []
vistos = set()
trial = 0
rng = np.random.default_rng(SEED)
t0 = time.time()
print("Estado inicializado. Lanza la celda del bucle.")

Estado inicializado. Lanza la celda del bucle.


### Bucle de búsqueda
Para **parar**: botón **Interrupt (■)**. Para **reanudar** donde lo dejaste: vuelve a ejecutar esta celda
(no reinicialices el estado). El leaderboard se imprime al parar y queda en la variable `leaderboard`.

In [8]:
try:
    while True:
        if MAX_TRIALS is not None and trial >= MAX_TRIALS:
            print(f"[fin] alcanzado MAX_TRIALS={MAX_TRIALS}."); break
        # --- dos cajas: k y k columnas, sin repetir el conjunto ---
        for _ in range(MAX_RESAMPLE):
            k = int(rng.integers(K_MIN, K_MAX + 1))
            cols = list(rng.choice(pool, size=k, replace=False))
            h = hash_conjunto(cols)
            if h not in vistos:
                vistos.add(h); break
        else:
            print("[fin] no encuentro conjuntos nuevos (espacio agotado)."); break
        trial += 1
        # --- secuencias del subconjunto (una vez para las 5 redes) ---
        idx = [col_idx[c] for c in cols]
        Xseq, yseq = construir_secuencias(X[:, idx], y, SEQ_LEN)
        Xtr, ytr = Xseq[:n_train], yseq[:n_train]
        Xva, yva = Xseq[n_train:n_train + n_val], yseq[n_train:n_train + n_val]
        # --- 5 redes ---
        vlosses, diraccs = [], []
        for red in REDES:
            vloss, diracc, nparams, bep = entrenar_red(red, Xtr, ytr, Xva, yva, tscaler)
            vlosses.append(vloss); diraccs.append(diracc)
            guardar_fila([datetime.now().isoformat(timespec="seconds"), trial, h, k,
                          red["id"], red["name"], nparams,
                          f"{vloss:.6f}", f"{diracc:.4f}", bep, ";".join(cols)])
        vlosses = np.array(vlosses)
        actualizar_leaderboard(leaderboard, {
            "k": k, "mean_val_loss": float(vlosses.mean()), "std_val_loss": float(vlosses.std()),
            "min_val_loss": float(vlosses.min()), "best_net": REDES[int(vlosses.argmin())]["name"],
            "mean_diracc": float(np.nanmean(diraccs)), "columnas": cols})
        if trial % PROGRESS_EVERY == 0:
            dt = time.time() - t0
            print(f"[trial {trial}] {dt/60:.1f} min | {trial/dt*3600:.0f} trials/h | "
                  f"mejor val_loss medio = {leaderboard[0]['mean_val_loss']:.6f} "
                  f"(k={leaderboard[0]['k']}, {leaderboard[0]['best_net']})", flush=True)
except KeyboardInterrupt:
    print("\n[interrumpido] leaderboard actual:")
finally:
    print(f"[resumen] {trial} subconjuntos probados en {(time.time()-t0)/60:.1f} min.")
    imprimir_leaderboard(leaderboard)


[interrumpido] leaderboard actual:
[resumen] 2 subconjuntos probados en 0.4 min.

  LEADERBOARD — top 1 por VAL_LOSS MEDIO entre las 5 redes
  (std bajo = rinde parecido en todas las redes = robusto)
   #    k  val_loss medio      std  mejor red  val_loss min  DirAcc med
  ------------------------------------------------------------------------------------------------
   1   14        0.260781   0.0011     maxima      0.259822      0.5169

  Columnas del nº1 (k=14):
     - alt_dominance_diff
     - btc_stoch_k
     - eth_cum_ret_30d
     - fear_greed_scaled
     - fg_presion_60d
     - fng_Greed
     - fng_Neutral
     - n_codicia_30d
     - n_codicia_ext_30d
     - n_miedo_ext_15d
     - n_miedo_ext_30d
     - n_neutral_15d
     - presion_ext_neta_90d
     - regime_2



### Análisis del leaderboard
La pregunta clave: **¿los buenos subconjuntos comparten columnas?** Si una columna aparece en casi todos
los mejores, eso es señal (no azar). Cuenta su frecuencia entre los top.

In [ ]:
from collections import Counter

imprimir_leaderboard(leaderboard)

# Frecuencia de cada columna entre los TOP subconjuntos del leaderboard
freq = Counter()
for r in leaderboard:
    freq.update(r["columnas"])
print(f"Columnas más frecuentes entre los {len(leaderboard)} mejores subconjuntos:\n")
for col, c in freq.most_common(25):
    barra = "#" * c
    print(f"  {c:>3}/{len(leaderboard):<3} {barra:<25} {col}")

### Utilidad: reconstruir el leaderboard desde el CSV
Por si reinicias el kernel o quieres mirar resultados de una corrida anterior.

In [ ]:
def reconstruir_leaderboard_desde_csv():
    df = pd.read_csv(OUT_DETALLE)
    lb = []
    for h, g in df.groupby("set_hash"):
        lb.append({"k": int(g.iloc[0]["k"]),
                   "mean_val_loss": g["val_loss"].mean(),
                   "std_val_loss": g["val_loss"].std(ddof=0),
                   "min_val_loss": g["val_loss"].min(),
                   "best_net": g.loc[g["val_loss"].idxmin(), "red_name"],
                   "mean_diracc": g["val_diracc"].mean(),
                   "columnas": g.iloc[0]["columnas"].split(";")})
    lb.sort(key=lambda r: r["mean_val_loss"]); del lb[TOP_K:]
    imprimir_leaderboard(lb)
    return lb

# lb = reconstruir_leaderboard_desde_csv()